<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z318_HAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAR  (Heterogeneous AutoRegressive)

## ¿Por qué HAR?

Los modelos de series de tiempo clásicos (ARIMA, ETS) asumen que el pasado inmediato es el predictor más relevante. HAR parte de una observación empírica distinta: en muchas series reales, lo que pasó **la semana pasada**, **el mes pasado** y **el año pasado** contribuyen de forma independiente y con distinto peso.

La ecuación central del modelo es:

```
tn_t = c  +  β1 · tn_{t-1}           ← señal del mes anterior
           +  β3 · mean(t-3 a t-1)   ← tendencia del trimestre
           +  β6 · mean(t-6 a t-1)   ← tendencia del semestre
           +  β12· mean(t-12 a t-1)  ← nivel del año
           +  ε
```

Es una **regresion lineal simple** — nada de cajas negras. Los coeficientes β son interpretables:
- Si β12 domina → las ventas del producto son estables año a año (baja volatilidad)
- Si β1 domina → hay mucha variacion mes a mes (alta volatilidad de corto plazo)
- Si todo β ≈ 0 → la serie es esencialmente ruido, ningun horizonte predice

## ¿Por qué no VAR?

VAR (Vector AutoRegression) modela todas las series juntas, capturando correlaciones cruzadas entre productos. Con 780 productos y 36 observaciones, un VAR completo necesitaria estimar 780² = 608.400 parametros — completamente inviable. HAR por producto es la alternativa practica y directamente interpretable.

## Estructura del notebook

1. **Test de Racha** por producto: ¿tiene estructura temporal detectable o es ruido puro?
2. **Ajuste HAR** por producto: regresion con los 4 horizontes
3. **Prediccion a dos pasos**: primero `202001`, luego `202002`
4. **Submit a Kaggle**

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo HAR

## 1.1 Init Experimento

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from scipy.stats import runstest_1samp

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia

In [ ]:
PARAM = {
  'experimento': 'HAR-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'alpha_runs': 0.05   # nivel de significancia del Test de Racha
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

In [ ]:
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner")
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

display(tb_ventas)

## 1.3 Test de Racha por producto

### ¿Por qué hacer esto antes de ajustar HAR?

HAR asume que la serie tiene **estructura temporal aprovechable** — que el pasado predice el futuro. Pero si una serie es esencialmente ruido aleatorio, ajustar HAR sobre ella no tiene sentido: el modelo va a sobreajustar sobre el ruido y predecir mal.

El **Test de Racha (Wald-Wolfowitz runs test)** es el chequeo previo adecuado:

- Toma la serie y cuenta las **rachas**: secuencias consecutivas de valores por encima o por debajo de la mediana
- **H₀**: la serie es aleatoria (sin estructura temporal)
- **H₁**: la serie tiene estructura (autocorrelacion positiva = pocas rachas, o negativa = muchas rachas)
- Si p-value < α → rechazamos H₀ → **hay estructura**, tiene sentido usar HAR
- Si p-value ≥ α → no podemos rechazar H₀ → la serie **parece ruido**, HAR probablemente no ayude

### ¿Qué hacemos con los productos "ruido"?

No los descartamos — igual los predecimos. Pero para ellos el fallback sera el **promedio de los ultimos 12 meses**, que es lo mejor que se puede hacer cuando no hay estructura temporal detectable.

In [ ]:
productos = tb_apredecir["product_id"].to_list()

runs_resultados = []

for producto in productos:
    serie = (
        tb_ventas
        .filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"]
        .to_numpy()
        .astype(float)
    )

    try:
        # runstest_1samp testea aleatoriedad contra la mediana de la serie
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = pvalue < PARAM['alpha_runs']
    except Exception:
        pvalue = np.nan
        tiene_estructura = False

    runs_resultados.append({
        'product_id': producto,
        'pvalue_runs': round(pvalue, 4) if not np.isnan(pvalue) else None,
        'tiene_estructura': tiene_estructura
    })

tb_runs = pl.DataFrame(runs_resultados)
display(tb_runs)

In [ ]:
# resumen: ¿cuantos productos tienen estructura temporal detectable?
n_total      = len(productos)
n_estructura = tb_runs["tiene_estructura"].sum()
n_ruido      = n_total - n_estructura

print(f"Total productos   : {n_total}")
print(f"Con estructura    : {n_estructura}  ({100*n_estructura/n_total:.1f}%)  → HAR tiene sentido")
print(f"Sin estructura    : {n_ruido}  ({100*n_ruido/n_total:.1f}%)  → fallback promedio 12m")

## 1.4 Construccion de features HAR

Para cada tiempo `t` se construyen cuatro features a partir del historial de la serie:

| Feature | Formula | Captura |
|---|---|---|
| `lag1` | `tn_{t-1}` | impulso del mes anterior |
| `mean_lag3` | `mean(tn_{t-3:t-1})` | tendencia del trimestre |
| `mean_lag6` | `mean(tn_{t-6:t-1})` | tendencia del semestre |
| `mean_lag12` | `mean(tn_{t-12:t-1})` | nivel base anual |

Se necesitan al menos 12 observaciones pasadas para construir todas las features, por lo tanto el entrenamiento arranca desde la observacion 13.

In [ ]:
def build_har_features(serie: np.ndarray):
    T = len(serie)
    rows_X, rows_y = [], []

    for t in range(12, T):
        lag1   = serie[t - 1]
        mean3  = serie[t-3 : t].mean()
        mean6  = serie[t-6 : t].mean()
        mean12 = serie[t-12: t].mean()
        rows_X.append([lag1, mean3, mean6, mean12])
        rows_y.append(serie[t])

    return np.array(rows_X), np.array(rows_y)


def har_predict_next(modelo, serie: np.ndarray) -> float:
    t = len(serie)
    lag1   = serie[t - 1]
    mean3  = serie[t-3 : t].mean()
    mean6  = serie[t-6 : t].mean()
    mean12 = serie[t-12: t].mean()
    X_pred = np.array([[lag1, mean3, mean6, mean12]])
    return float(modelo.predict(X_pred)[0])

## 1.5 Ajuste HAR por producto y prediccion a dos pasos

### ¿Por qué dos pasos?

El objetivo es predecir `202002` (febrero 2020), pero el ultimo dato conocido es `201912` (diciembre 2019). Hay un gap de dos meses.

HAR usa como input el mes anterior y promedios recientes — todos valores conocidos para predecir `202001`. Para predecir `202002` necesitamos `202001`, que no existe en los datos. La solucion es:

1. **Paso 1**: predecir `202001` usando los 36 meses reales como historia
2. **Paso 2**: agregar `pred_202001` al final de la serie y predecir `202002` con esa serie extendida

Este approach se llama **recursive forecasting** — el modelo se usa iterativamente sobre sus propias predicciones.

### Estrategia por tipo de producto

- **Con estructura** (Test de Racha rechazó H₀): se ajusta HAR y se predice con el modelo
- **Sin estructura** (serie es ruido): fallback directo al promedio de los ultimos 12 meses, sin ajustar HAR

In [ ]:
# diccionario rapido: product_id -> tiene_estructura
estructura_dict = dict(zip(
    tb_runs["product_id"].to_list(),
    tb_runs["tiene_estructura"].to_list()
))

resultados = []

for producto in productos:
    print(producto, end=' ')

    serie = (
        tb_ventas
        .filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"]
        .to_numpy()
        .astype(float)
    )

    fallback = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    if not estructura_dict.get(producto, False):
        # serie sin estructura detectable → promedio 12m es lo mejor
        pred_2 = fallback
        metodo = 'promedio12m'
    else:
        try:
            X, y = build_har_features(serie)
            modelo = LinearRegression()
            modelo.fit(X, y)

            # paso 1: predecir 202001
            pred_1 = max(har_predict_next(modelo, serie), 0.0)

            # paso 2: predecir 202002
            serie_ext = np.append(serie, pred_1)
            pred_2 = max(har_predict_next(modelo, serie_ext), 0.0)
            metodo = 'HAR'

        except Exception as e:
            print(f"  ERROR {producto}: {e}")
            pred_2 = fallback
            metodo = 'promedio12m_error'

    resultados.append({'product_id': producto, 'tn': pred_2, 'metodo': metodo})

print("\nlisto")

## 1.6 Armado del submit

In [ ]:
tb_resultado = pl.DataFrame(resultados)

# resumen de metodos usados
display(tb_resultado.group_by("metodo").agg(pl.len().alias("n_productos")))

# tabla final solo con product_id y tn
tb_final = tb_resultado.select(["product_id", "tn"])
display(tb_final)
print("nulls:", tb_final["tn"].is_null().sum())

## 1.7 Submit a Kaggle

In [ ]:
archivo = "HAR.csv"
mensaje = "HAR lag1-3-6-12 dos pasos con Test de Racha"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)